# Clip --

**CLIP (Contrastive Language-Image Pre-training)** is a model developed by OpenAI that learns to connect images and natural-language text.- It is trained on (image, caption) pairs using **contrastive learning**: the model learns to pull the embeddings of a matching image-text pair close together in a shared vector space, while pushing non-matching pairs apart.- Because images and text end up in the *same* embedding space, CLIP can perform **zero-shot classification** — you can classify an image into any set of text labels without additional training, just by comparing embeddings.- Common uses: zero-shot image classification, image-text retrieval, and as a building block inside larger multimodal models (e.g. guiding image generation in models like DALL·E and Stable Diffusion).

# What are AI benchmarks?

**AI benchmarks** are standardized datasets, tasks, and metrics used to evaluate and compare the performance of AI models under the same conditions.- They provide a common yardstick so that different models/approaches can be compared fairly (e.g. accuracy, F1-score, BLEU, perplexity).- Examples:  - **GLUE / SuperGLUE** – general language understanding tasks (NLP)  - **MMLU** – broad multitask knowledge and reasoning across 57 subjects  - **ImageNet** – large-scale image classification  - **HellaSwag, TruthfulQA, HumanEval** – commonsense reasoning, truthfulness, and code generation respectively- Benchmarks help track progress over time and reveal a model's strengths/weaknesses, but they can also be gamed (models overfitting to benchmark-style data), so results should be interpreted with some caution.

# What is the difference framework and library?

**Library vs Framework** — the key difference is about *who is in control* (a concept called *inversion of control*):- **Library**: a collection of reusable functions/classes that *you* call from your own code, whenever and however you like. You are in control of the program flow. Example: `requests`, `NumPy`, `pandas`.- **Framework**: provides the overall skeleton/structure of an application; *it* calls *your* code (via hooks, callbacks, or configuration) rather than the other way around. Example: `Flask`/`Django` (web), `PyTorch Lightning` (training loop structure).In short: **you call a library; a framework calls you.**

## Loading a model with 8-bit Quantization (e.g. loading weights in 8-bit instead of 32-bit) reduces the memory footprint of large models with minimal accuracy loss, using the `bitsandbytes` library via `transformers`.

In [ ]:
!pip install --upgrade -q transformers accelerate "bitsandbytes>=0.46.1"

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(load_in_8bit=True)

model_8bit = AutoModelForCausalLM.from_pretrained(
    "bigscience/bloom-1b7",
    device_map = "auto",
    quantization_config= quantization_config
)

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    pipeline
)

model_name ="bigscience/bloom-1b7"

quantization_config = BitsAndBytesConfig(load_in_8bit=True)

tokenizer = AutoTokenizer.from_pretrained(model_name)

pipe = pipeline(
    "text-generation",
    model = model_8bit,
    tokenizer = tokenizer
)

result = pipe(
    "what is machine learning?",
    max_new_tokens=100
)

print(result[0]["generated_text"])
model_8bit

tokenizer_config.json:   0%|          | 0.00/222 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


KeyboardInterrupt: 

# Cloudflare--

[Cloudflare Workers AI](https://developers.cloudflare.com/workers-ai/) lets you run inference on hosted open-source models (like Llama 3.1) through a simple REST API, without managing any GPU infrastructure yourself. You just need an `ACCOUNT_ID` and an API token (`CF_TOKEN`) stored as Colab secrets.

In [ ]:
import requests
from google.colab import userdata
ACCOUNT_ID = userdata.get('ACCOUNT_ID')
CF_TOKEN = userdata.get('CF_TOKEN')

def chat(message):
  url = f"https://api.cloudflare.com/client/v4/accounts/{ACCOUNT_ID}/ai/run/@cf/meta/llama-3.1-8b-instruct"

  headers = {
      "Authorization": f"Bearer {CF_TOKEN}",
      "Content-Type": "application/json"
  }

  payload = {
      "messages": [
          {
              "role": "user",
              "content": message
          }
      ]
  }
  response = requests.post(url, headers=headers, json=payload)

  print("Status Code:", response.status_code)

  if response.status_code != 200:
    print(response.text)
    return None

  data = response.json()
  return data["result"]["response"]
chat("hello, i am testing your response")

Status Code: 200


"Hello.  I'm happy to assist you with any questions or tasks you may have. What would you like to test or discuss?"

# Ragas--
     To analyse the performance of LLMs.

**Context Precision** — of the chunks retrieved, how many are actually relevant to the question?`Precision = (number of retrieved chunks that are relevant) / (total number of chunks retrieved) * 100`

**Context Recall** — of all the chunks that are actually relevant (anywhere in the database), how many did we successfully retrieve?`Recall = (number of retrieved chunks that are relevant) / (total number of relevant chunks in the entire database) * 100`> **Correction:** the denominator is the total number of *relevant* chunks in the database — not the total number of chunks in the database overall.